# Option B Exploration

This notebook contains the exploration component for Option B, due the week after the main notebook. Choose something related to the topics covered in the main notebook and build something creative or interesting.

_For this notebook some helpful **starting points** for the extension include:_

* **Stereo view synthesis:** Start from a **dense disparity map** (e.g. the motorcycle stereo pair in Part 2) and aim to **draw the scene again** as if a **second, shifted camera** had taken the picture- not just a false-color disparity plot.

* **Metric depth:** Turn disparities into **metric depth** with $\text{depth} = \frac{B \cdot f}{\text{disparity}}$ using a plausible **baseline** and **focal length**, and stabilize the denominator when disparity is tiny or noisy.

* **Calibration & lifting:** Use the camera **intrinsics** ($3 \times 3$ matrix $K$) together with disparity and baseline to **lift pixels from the image into 3D** in the original camera’s coordinate frame.

* **Virtual camera motion:** Describe how the new camera sits relative to the first with a **rigid motion**—a **rotation** and **translation**—and **move every 3D point** into that new camera’s frame before drawing.

* **Rendering:** **Project** those 3D points back onto a **fresh image grid** with intrinsics suited to the new view, producing the **rerendered photograph** (including how you handle resolution, bounds, and empty pixels).

For more detailed instructions, please see the [Project 2 Guidelines](https://docs.google.com/document/d/1_w5RefFChBW3IdvhiCvtLgPGkIDNeWNKhVrLjKPL2Fw/edit?usp=sharing)


## Review of current methods (10 points)
Once you’ve selected a topic or project idea, explore the literature space. Has there been academic research on this topic? Are there tutorials online, software packages, or libraries?

Select at least 5 resources (youtube videos, papers, tutorials, opensource software, libraries, etc) and provide a short description (2-3 sentences) below:

* Source 1:
* Source 2:
* Source 3:
* Source 4:
* Source 5:

## Code (70 points)
We expect you to write code for this project (CS131 is, after all, a CS class 🙂). You may implement algorithms from scratch or expand on algorithms from this notebook if you would like, but using other libraries or other open-source software in a creative way is also sufficient.

You are not required to develop your code in this notebook! Feel free to create your own jupyter notebook for the project or write code in your environment of choice! (Jupyter notebook or google colab are good starting options)!


In [17]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
from skimage import filters
from skimage.feature import corner_peaks
from scipy.ndimage import convolve

In [18]:


def harris_corners(img, window_size=3, k=0.04):

    H, W = img.shape
    window = np.ones((window_size, window_size))

    response = np.zeros((H, W))

    # 1. Compute x and y derivatives (I_x, I_y) of an image
    dx = filters.sobel_v(img)
    dy = filters.sobel_h(img)
    Ixx = dx * dx
    Iyy = dy * dy
    Ixy = dx * dy

    Sxx = convolve(Ixx, window, mode='constant', cval=0)
    Syy = convolve(Iyy, window, mode='constant', cval=0)
    Sxy = convolve(Ixy, window, mode='constant', cval=0)

    det_M = Sxx * Syy - Sxy * Sxy
    trace_M = Sxx + Syy

    response = det_M - k * (trace_M ** 2)
    return response


In [19]:

img_left = cv.imread('motorcycle_left.png', cv.IMREAD_GRAYSCALE)
img_right = cv.imread('motorcycle_right.png', cv.IMREAD_GRAYSCALE)

stereo = cv.StereoSGBM_create(minDisparity=0, numDisparities=64, blockSize=5)
disparity = stereo.compute(img_left, img_right).astype(np.float32) / 16.0

H, W = disparity.shape
f = 3979.911 * (W / 2964)
B = 0.193


def compute_depth(disparity, threshold):
    depth = np.zeros_like(disparity)
    valid = disparity > threshold
    depth[valid] = (f * B) / disparity[valid]
    return depth, valid


thresholds = [1, 16, 32, 64]

plt.figure(figsize=(16, 4))

for i, t in enumerate(thresholds):
    depth, valid = compute_depth(disparity, t)

    display = depth.copy()
    display[~valid] = np.nan
    vmax = np.nanpercentile(display, 95)

    plt.subplot(1, 4, i+1)
    plt.imshow(display, cmap='viridis', vmin=0, vmax=vmax)
    plt.title(f"Threshold = {t}")
    plt.axis('off')

plt.suptitle("Effect of Disparity Threshold on Depth Stability")
plt.tight_layout()
plt.show()


depth, valid = compute_depth(disparity, 16)

img_float = img_left.astype(np.float32) / 255.0
keypoints = corner_peaks(harris_corners(img_float), threshold_rel=0.05, exclude_border=8)

kp_depths = []
kp_locs = []

for y, x in keypoints:
    if valid[y, x]:
        kp_depths.append(depth[y, x])
        kp_locs.append((y, x))

kp_depths = np.array(kp_depths)
kp_locs = np.array(kp_locs)

if len(kp_locs) > 0:
    plt.figure(figsize=(10, 5))
    plt.imshow(img_left, cmap='gray')
    plt.scatter(kp_locs[:, 1], kp_locs[:, 0],
                c=kp_depths, cmap='viridis', s=20, edgecolors='red')
    plt.colorbar(label='Depth (m)')
    plt.title('Harris Corners Colored by Metric Depth')
    plt.axis('off')
    plt.show()

error: OpenCV(4.13.0) /io/opencv/modules/calib3d/src/stereosgbm.cpp:511: error: (-2:Unspecified error) in function 'void cv::computeDisparitySGBM(const Mat&, const Mat&, Mat&, const StereoSGBMParams&)'
> Your input images are too small for your window size and max disparity, and will result in non-deterministic SGBM results (expected: 'width - (params.minDisparity + params.numDisparities) > params.calcSADWindowSize().width/2'), where
>     'width - (params.minDisparity + params.numDisparities)' is -144
> must be greater than
>     'params.calcSADWindowSize().width/2' is 3


## Writeup (20 points)

An explanation of what you did, and how it relates to the topic of choice. (~200 words) Please attach any images, figures, etc.

_You may also add a link to your writeup if that is easier!_